# ETS Thermodynamic Model Verification (V1)

Controlled experiment: the actual `ETSModelV1.step()` is called with fixed load and supply temperature while only the flow command is varied. All table values are outputs of the implementation (nothing is hand-calculated); Pass/Fail is checked against the governing equations.

Note: the calibrated ETS capacity (`max_flow * Cp * design dT`) is only ~15 kW, so a 100 kW load always exceeds it. The main sweep therefore uses a load that fits within capacity (`LOAD_KW`), and 100 kW is kept as an overload test.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd()))
from ets_model_v1 import ETSModelV1, calibrate_from_energyplus, CP_WATER

# Locate the EnergyPlus CSV used for calibration
CSV = next(p / "eplusout.csv" for p in [Path.cwd(), *Path.cwd().parents] if (p / "eplusout.csv").exists())
DESIGN_DT = 4.0
SUPPLY_T = 6.0
LOAD_KW = 7.0  # constant demand within ETS capacity

cfg = calibrate_from_energyplus(pd.read_csv(CSV), DESIGN_DT)
model = ETSModelV1(cfg)
print(f"CSV: {CSV}")
print(f"Max flow {cfg.max_flow_kg_s:.4f} kg/s | capacity at action 1.0 = {cfg.max_flow_kg_s*CP_WATER*DESIGN_DT/1000:.2f} kW")

CSV: /Users/ashanperera/EnergyPlus/RL_UAE_HVAC_SOFTWARE/eplusout.csv
Max flow 0.9077 kg/s | capacity at action 1.0 = 15.18 kW


In [2]:
test_cases = [
    {"name": "Low flow",      "load_kw": LOAD_KW, "action": 0.20},
    {"name": "Normal flow",   "load_kw": LOAD_KW, "action": 0.50},
    {"name": "High flow",     "load_kw": LOAD_KW, "action": 0.80},
    {"name": "Max flow",      "load_kw": LOAD_KW, "action": 1.00},
    {"name": "Zero load",     "load_kw": 0.0,     "action": 0.50},
    {"name": "Zero flow",     "load_kw": LOAD_KW, "action": 0.00},
    {"name": "Overload 100 kW", "load_kw": 100.0,  "action": 1.00},
]

def run_case(tc):
    r = model.step(tc["load_kw"] * 1000.0, SUPPLY_T, tc["action"])
    return {
        "Test": tc["name"],
        "Load (kW)": tc["load_kw"],
        "Action": tc["action"],
        "Flow (kg/s)": r.flow_kg_s,
        "Ts (C)": r.supply_temp_c,
        "Tr (C)": r.return_temp_c,
        "dT (C)": r.delta_t_c,
        "Capacity (kW)": r.q_capacity_w / 1000,
        "Delivered (kW)": r.q_delivered_w / 1000,
        "Unmet (kW)": r.q_unmet_w / 1000,
        "Pump (W)": r.pump_power_w,
    }

res = pd.DataFrame([run_case(tc) for tc in test_cases]).set_index("Test")
res

,Load (kW),Action,Flow (kg/s),Ts (C),Tr (C),dT (C),Capacity (kW),Delivered (kW),Unmet (kW),Pump (W)
Test,,,,,,,,,,
Low flow,7.0,0.2,0.181540,6.0,10.000000,4.000000,3.035356,3.035356,3.964644,46.394050
Normal flow,7.0,0.5,0.453851,6.0,9.689847,3.689847,7.588391,7.000000,0.000000,115.985125
High flow,7.0,0.8,0.726162,6.0,8.306154,2.306154,12.141425,7.000000,0.000000,185.576199
Max flow,7.0,1.0,0.907702,6.0,7.844923,1.844923,15.176781,7.000000,0.000000,231.970249
Zero load,0.0,0.5,0.453851,6.0,6.000000,0.000000,7.588391,0.000000,0.000000,115.985125
Zero flow,7.0,0.0,0.000000,6.0,6.000000,0.000000,0.000000,0.000000,7.000000,0.000000
Overload 100 kW,100.0,1.0,0.907702,6.0,10.000000,4.000000,15.176781,15.176781,84.823219,231.970249


In [3]:
tol = 1e-6

def check(row):
    q, d, u = row["Load (kW)"], row["Delivered (kW)"], row["Unmet (kW)"]
    m, dt = row["Flow (kg/s)"], row["dT (C)"]
    ok = abs(d + u - q) < tol                                   # energy balance
    ok &= d <= row["Capacity (kW)"] + tol and d <= q + tol      # constraints
    ok &= abs(row["Tr (C)"] - row["Ts (C)"] - dt) < tol         # Tr = Ts + dT
    ok &= dt <= DESIGN_DT + tol and np.isfinite(dt)             # bounded, stable
    if m > 1e-9:
        ok &= abs(dt - d * 1000 / (m * CP_WATER)) < 1e-6        # dT = Q/(m Cp)
    return "Pass" if ok else "Fail"

res["Result"] = res.apply(check, axis=1)

# Trend test: constant load, flow up -> dT down (while load fully delivered)
trend = res.loc[["Normal flow", "High flow", "Max flow"], "dT (C)"].to_numpy()
trend_ok = bool(np.all(np.diff(trend) < 0))
# Low flow must be capacity-limited (unmet > 0), dT at design value
low_ok = res.loc["Low flow", "Unmet (kW)"] > 0 and abs(res.loc["Low flow", "dT (C)"] - DESIGN_DT) < 1e-6
print("dT decreases as flow increases (Normal > High > Max):", "Pass" if trend_ok else "Fail")
print("Low flow is capacity-limited with unmet cooling     :", "Pass" if low_ok else "Fail")
res.round(4)

dT decreases as flow increases (Normal > High > Max): Pass
Low flow is capacity-limited with unmet cooling     : Pass


,Load (kW),Action,Flow (kg/s),Ts (C),Tr (C),dT (C),Capacity (kW),Delivered (kW),Unmet (kW),Pump (W),Result
Test,,,,,,,,,,,
Low flow,7.0,0.2,0.1815,6.0,10.0000,4.0000,3.0354,3.0354,3.9646,46.3940,Pass
Normal flow,7.0,0.5,0.4539,6.0,9.6898,3.6898,7.5884,7.0000,0.0000,115.9851,Pass
High flow,7.0,0.8,0.7262,6.0,8.3062,2.3062,12.1414,7.0000,0.0000,185.5762,Pass
Max flow,7.0,1.0,0.9077,6.0,7.8449,1.8449,15.1768,7.0000,0.0000,231.9702,Pass
Zero load,0.0,0.5,0.4539,6.0,6.0000,0.0000,7.5884,0.0000,0.0000,115.9851,Pass
Zero flow,7.0,0.0,0.0000,6.0,6.0000,0.0000,0.0000,0.0000,7.0000,0.0000,Pass
Overload 100 kW,100.0,1.0,0.9077,6.0,10.0000,4.0000,15.1768,15.1768,84.8232,231.9702,Pass


In [4]:
out = Path("ets_verification_results.csv")
res.round(4).to_csv(out)
print("Saved", out.resolve())
print(res.round(4).to_string())

Saved /Users/ashanperera/EnergyPlus/RL_UAE_HVAC_SOFTWARE/V1_controllable offline ETS model/src/ets_verification_results.csv
                 Load (kW)  Action  Flow (kg/s)  Ts (C)   Tr (C)  dT (C)  Capacity (kW)  Delivered (kW)  Unmet (kW)  Pump (W) Result
Test                                                                                                                                
Low flow               7.0     0.2       0.1815     6.0  10.0000  4.0000         3.0354          3.0354      3.9646   46.3940   Pass
Normal flow            7.0     0.5       0.4539     6.0   9.6898  3.6898         7.5884          7.0000      0.0000  115.9851   Pass
High flow              7.0     0.8       0.7262     6.0   8.3062  2.3062        12.1414          7.0000      0.0000  185.5762   Pass
Max flow               7.0     1.0       0.9077     6.0   7.8449  1.8449        15.1768          7.0000      0.0000  231.9702   Pass
Zero load              0.0     0.5       0.4539     6.0   6.0000  0.0000      